# 02 — Evaluate Qwen2.5-72B-Instruct

Zero-/few-shot evaluation on the splits made by `01_data_split.ipynb`.

**Two backends:**
- `api` — any OpenAI-compatible endpoint (OpenRouter, Together, DashScope, or your own vLLM server). Runs on a laptop with no GPU. Put the key in `.env` (see `.env.example`).
- `vllm` — loads the model on a local GPU (Linux). The 72B model needs ~2×80GB in bf16, or one 80GB GPU with the AWQ build `Qwen/Qwen2.5-72B-Instruct-AWQ`.

Predictions are written as they arrive, so if the run is interrupted, just run the cells again and it picks up where it stopped.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

## Configuration — edit this cell

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

BACKEND  = "api"            # "api" or "vllm"
SPLITS   = ["val", "test"]  # which splits to evaluate
N_SHOTS  = 0                # few-shot examples taken from the train split (same subject); try 0 and 5
LIMIT    = None             # e.g. 20 for a quick smoke test, None for the full split
SEED     = 42

# --- api backend ---
API_MODEL    = os.getenv("MODEL_NAME", "qwen/qwen-2.5-72b-instruct")
API_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")
API_KEY      = os.getenv("OPENAI_API_KEY")
MAX_WORKERS  = 8            # parallel requests; lower it if you get rate-limited

# --- vllm backend ---
VLLM_MODEL   = "Qwen/Qwen2.5-72B-Instruct-AWQ"   # or "Qwen/Qwen2.5-72B-Instruct" with 2+ big GPUs
VLLM_TP      = 1            # tensor_parallel_size = number of GPUs
VLLM_QUANT   = None         # vLLM detects AWQ automatically

RUN_NAME = f"qwen2.5-72b-instruct_{BACKEND}_{N_SHOTS}shot"
RUN_DIR  = ROOT / "results" / RUN_NAME
print("Results ->", RUN_DIR)

## Load splits

In [ ]:
from src.data import load_jsonl

SPLIT_DIR = ROOT / "data" / "splits"
data = {s: load_jsonl(SPLIT_DIR / f"{s}.jsonl") for s in ["train", "val", "test"]}
for s, d in data.items():
    print(f"{s:5s}: {len(d)}")

## Check the prompt

In [ ]:
from src.prompting import ShotSampler, build_messages

sampler = ShotSampler(data["train"], k=N_SHOTS, seed=SEED)
example = data["test"].iloc[0].to_dict()
for m in build_messages(example, sampler(example)):
    print(f"--- {m['role']} ---\n{m['content']}\n")
print("Gold answer:", example["answer"])

## Start the backend

In [ ]:
from src.backends import OpenAICompatBackend, VLLMBackend

if BACKEND == "api":
    assert API_KEY, "Set OPENAI_API_KEY in .env"
    backend = OpenAICompatBackend(API_MODEL, API_BASE_URL, API_KEY, max_workers=MAX_WORKERS)
    BATCH_SIZE = MAX_WORKERS * 4
elif BACKEND == "vllm":
    backend = VLLMBackend(VLLM_MODEL, tensor_parallel_size=VLLM_TP, quantization=VLLM_QUANT)
    BATCH_SIZE = 256
else:
    raise ValueError(BACKEND)

# Sanity check with one question
print(repr(backend.generate([build_messages(example, sampler(example))])[0]))

## Run evaluation

In [ ]:
from src.evaluate import run_eval, compute_metrics, save_metrics

all_metrics = {}
for split_name in SPLITS:
    df = data[split_name]
    if LIMIT:
        df = df.sample(n=min(LIMIT, len(df)), random_state=SEED)
    print(f"\n=== {split_name} ({len(df)} questions) ===")
    preds = run_eval(df, backend, sampler, RUN_DIR / f"{split_name}_predictions.jsonl", batch_size=BATCH_SIZE)
    m = compute_metrics(preds)
    save_metrics(m, RUN_DIR / f"{split_name}_metrics.json")
    all_metrics[split_name] = m
    print(f"accuracy = {m['accuracy']:.2%}   unparsed = {m['unparsed_rate']:.2%}   n = {m['n']}")

## Results

In [ ]:
import pandas as pd

summary = pd.DataFrame({s: {"n": m["n"], "accuracy": m["accuracy"], "unparsed_rate": m["unparsed_rate"]}
                        for s, m in all_metrics.items()}).T
summary.to_csv(RUN_DIR / "summary.csv")
summary

In [ ]:
# Accuracy per subject, side by side for each evaluated split
pd.concat({s: m["by_subject"] for s, m in all_metrics.items()}, axis=1)

In [ ]:
pd.concat({s: m["by_category"] for s, m in all_metrics.items()}, axis=1)

## Error inspection

In [ ]:
# Outputs the parser could not read: if there are many, adjust the prompt or src/prompting.parse_answer
split_name = SPLITS[-1]
preds = pd.read_json(RUN_DIR / f"{split_name}_predictions.jsonl", lines=True)
preds[preds["pred"].isna()][["id", "raw_output"]].head(20)

In [ ]:
# Wrong answers, joined with the question text
wrong = preds[~preds["correct"]].merge(data[split_name][["id", "question", "choices"]], on="id")
wrong[["subject", "question", "choices", "answer", "pred", "raw_output"]].head(20)